# DepthWizard — Photo to Fake DSM (GeoTIFF)

This notebook turns a single 2D photo into a **GeoTIFF elevation raster** using monocular depth estimation (MiDaS).

**Important:** the output elevation values are **not real-world calibrated**. MiDaS produces *relative* depth (near/far), which is rescaled here to an arbitrary 0-100 m range and placed at a placeholder geographic location. This is useful for testing the file pipeline (GeoTIFF format, CRS, visualization) but not for actual elevation analysis. To get real elevation you'd need calibration against ground control points or a reference DEM (e.g. SRTM/Copernicus DEM), plus a real geographic anchor (e.g. from photo GPS EXIF).

**Steps:**
1. Install dependencies
2. Upload a photo
3. Run MiDaS depth estimation
4. Rescale depth -> fake elevation and save as GeoTIFF
5. Verify by reloading and plotting

## 1. Install dependencies

In [ ]:
!pip install rasterio timm -q

import numpy as np
import torch
import cv2
import rasterio
from rasterio.transform import from_origin
import matplotlib.pyplot as plt

print("Dependencies ready.")

## 2. Upload a photo

Run this cell and pick an image file (JPG/PNG) from your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print("Uploaded:", image_path)

## 3. Run MiDaS depth estimation

Loads the small/fast MiDaS model and predicts a relative depth map for the uploaded image.

In [ ]:
# trust_repo=True skips the interactive "do you trust this repo" prompt
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small", trust_repo=True)
midas.eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms", trust_repo=True)
transform = midas_transforms.small_transform

img = cv2.imread(image_path)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

input_batch = transform(img)

print("Running depth model... (takes ~10-20 sec)")
with torch.no_grad():
    prediction = midas(input_batch)
    prediction = torch.nn.functional.interpolate(
        prediction.unsqueeze(1),
        size=img.shape[:2],
        mode="bicubic",
        align_corners=False,
    ).squeeze()

depth_map = prediction.cpu().numpy()

plt.figure(figsize=(6, 5))
plt.imshow(depth_map, cmap="inferno")
plt.colorbar(label="Relative depth")
plt.title("Predicted depth (relative, not real elevation)")
plt.show()

print("Depth map shape:", depth_map.shape)
print("Min:", depth_map.min(), "Max:", depth_map.max())

## 4. Rescale depth -> fake elevation, save as GeoTIFF

The lon/lat anchor below (77.4, 23.3) is a placeholder, not the photo's real location. Replace it with real coordinates (e.g. from the photo's GPS EXIF data) if you want the georeferencing to mean something.

In [ ]:
# Placeholder geographic anchor — replace with real coordinates if you have them
ORIGIN_LON, ORIGIN_LAT = 77.4, 23.3
PIXEL_SIZE_DEG = 0.0001
OUTPUT_FILENAME = "real_photo_dsm.tif"

# Normalize depth to an arbitrary 0-100 m range
# (Real calibration would use a reference DEM or ground control points instead)
elevation = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min()) * 100
elevation = elevation.astype("float32")

height, width = elevation.shape
transform_geo = from_origin(ORIGIN_LON, ORIGIN_LAT, PIXEL_SIZE_DEG, PIXEL_SIZE_DEG)

with rasterio.open(
    OUTPUT_FILENAME, "w",
    driver="GTiff",
    height=height, width=width,
    count=1, dtype="float32",
    crs="EPSG:4326",
    transform=transform_geo,
) as dst:
    dst.write(elevation, 1)

print(f"Saved {OUTPUT_FILENAME} (fake elevation model, not real-world calibrated)")

files.download(OUTPUT_FILENAME)

## 5. Verify the saved GeoTIFF

In [ ]:
with rasterio.open(OUTPUT_FILENAME) as src:
    data = src.read(1)
    print("CRS:", src.crs, "| min:", data.min(), "| max:", data.max())

plt.figure(figsize=(6, 5))
plt.imshow(data, cmap="terrain")
plt.colorbar(label="Elevation (m) — fake scale")
plt.title(f"{OUTPUT_FILENAME} contents")
plt.show()